In [1]:
MODEL_PATH = "runs/detect/train2/weights/best.pt"

In [2]:
import torch
# import torchvision.models as models 
from pytorch_nndct.apis import torch_quantizer


[VAIQ_NOTE]: Loading NNDCT kernels...


In [ ]:
# model = torch.load(MODEL_PATH)
# model.eval()  # 記得如果是推論，要設 eval 模式

from ultralytics import YOLO

# 載入 YOLO 模型
yolo_model = YOLO(MODEL_PATH)  # 或者你自己的.pt

# 取出原生 PyTorch 模型
torch_model = yolo_model.model
state_dict = torch_model.state_dict()

# 模型的通道數和輸入尺寸
print("Classes:", torch_model.yaml['nc'])  # 類別數量
print(torch_model)

Classes: 4
DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C3k2(
      (cv1): Conv(
        (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)


In [4]:
input_shape = (1, 3, 640, 640) 
dummy_input = torch.randn(input_shape)

# 3. 初始化 Vitis AI 量化器 (模式設為 'calib' 進行校準)
quantizer = torch_quantizer('calib', torch_model, (dummy_input), output_dir='quantize_result')
quant_model = quantizer.quant_model


[VAIQ_NOTE]: OS and CPU information:
               system --- Linux
                 node --- jianhua-MS-7D95
              release --- 6.8.0-106-generic
              version --- #106~22.04.1-Ubuntu SMP PREEMPT_DYNAMIC Fri Mar  6 08:44:59 UTC 
              machine --- x86_64
            processor --- x86_64

[VAIQ_NOTE]: Tools version information:
                  GCC --- GCC 7.5.0
               python --- 3.8.6
              pytorch --- 1.13.1+cu117
        vai_q_pytorch --- 3.5.0+60df3f1+torch1.13.1+cu117

[VAIQ_NOTE]: GPU information:
          device name --- NVIDIA GeForce RTX 4070
     device available --- True
         device count --- 1
       current device --- 0

[VAIQ_NOTE]: Quant config file is empty, use default quant configuration

[VAIQ_NOTE]: Quantization calibration process start up...

[VAIQ_NOTE]: =>Quant Module is in 'cuda'.

[VAIQ_NOTE]: =>Parsing DetectionModel...

[VAIQ_NOTE]: Start to trace and freeze model...

[VAIQ_NOTE]: The input model nndct_st_Detecti

██████████████████████████████████████████████████| 357/357 [00:00<00:00, 1011.88it/s, OpInfo: name = return_0, type = Return]                                                                                                                 


[VAIQ_WARN][QUANTIZER_TORCH_FLOAT_OP]: The quantizer recognize new op `aten::silu_` as a float operator by default.



[VAIQ_NOTE]: =>Doing weights equalization...

[VAIQ_NOTE]: =>Quantizable module is generated.(quantize_result/DetectionModel.py)

[VAIQ_NOTE]: =>Get module with quantization.


In [5]:
print("開始校準...")
with torch.no_grad():
    # for images, _ in calibration_loader:
    #     quant_model(images)
    quant_model(dummy_input) # 僅示範用，請換成真實數據迴圈

開始校準...

[VAIQ_WARN][QUANTIZER_TORCH_TENSOR_TYPE_NOT_QUANTIZABLE]: The tensor type of DetectionModel::DetectionModel/C2PSA[model]/C2PSA[10]/Sequential[m]/PSABlock[0]/Attention[attn]/ret.203 is torch.int64. Only support float32/double/float16 quantization.


/home/jianhua/miniconda3/envs/vai_pytorch/lib/python3.8/site-packages/pytorch_nndct/quantization/torchquantizer.py:223: FutureWarning: Unlike other reduction functions (e.g. `skew`, `kurtosis`), the default behavior of `mode` typically preserves the axis it acts along. In SciPy 1.11.0, this behavior will change: the default value of `keepdims` will become False, the `axis` over which the statistic is taken will be eliminated, and the value None will no longer be accepted. Set `keepdims` to True or False to avoid this warning.
  bnfp[1] = stats.mode(data)[0][0]


In [7]:
# 5. 匯出量化設定檔與中繼的 xmodel
print("匯出量化模型...")
quantizer.export_quant_config()
quantizer.export_xmodel(deploy_check=True, output_dir='quantize_result')

匯出量化模型...

[VAIQ_NOTE]: =>Exporting quant config.(quantize_result/quant_info.json)

[VAIQ_WARN][QUANTIZER_TORCH_TENSOR_NOT_QUANTIZED]: Node ouptut tensor is not quantized: DetectionModel::DetectionModel/C2PSA[model]/C2PSA[10]/Sequential[m]/PSABlock[0]/Attention[attn]/ret.203 type: nndct_elemwise_mul

[VAIQ_WARN][QUANTIZER_TORCH_TENSOR_NOT_QUANTIZED]: Some tensors are not quantized, please check their particularity.
